# 第 12 週 實作｜功、質心與期望值

積分不只算幾何量。物理的功與質心、機率的期望值與變異數,骨子裡是同一個積分結構——而這也是 <code>loss.mean()</code> 的真正身分。


### (選用)讓圖表顯示中文


In [ ]:
import matplotlib
# Colab 想顯示中文: !apt-get -qq install fonts-noto-cjk
# 再設 matplotlib.rcParams['font.sans-serif'] = ['Noto Sans CJK TC']
# 本課圖表標籤一律用英文,不裝字型也不會有豆腐字。
matplotlib.rcParams['axes.unicode_minus'] = False


### 環境設定


In [ ]:
import math
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt


## Lab 1｜期望值與變異數:三個分佈算到底

觀念 5、6 的公式,對三個常見分佈各算一次,並用大量取樣驗證。


In [ ]:
x = sp.Symbol('x', real=True)

dists = [
    ("均勻 U(0,1)",  sp.Integer(1),                              0, 1),
    ("指數 Exp(1)",  sp.exp(-x),                                 0, sp.oo),
    ("標準常態",      sp.exp(-x**2/2)/sp.sqrt(2*sp.pi),          -sp.oo, sp.oo),
]

print(f"{'分佈':14s} {'∫p (應為1)':>10} {'E[X]':>10} {'E[X^2]':>10} {'Var':>10}")
theory = {}
for name, p, a, b in dists:
    Z  = sp.integrate(p, (x, a, b))
    m1 = sp.integrate(x*p, (x, a, b))
    m2 = sp.integrate(x**2*p, (x, a, b))
    var = sp.simplify(m2 - m1**2)
    theory[name] = (float(m1), float(var))
    print(f"{name:14s} {str(Z):>10} {str(m1):>10} {str(m2):>10} {str(var):>10}")

# --- 用取樣驗證 ---
rng = np.random.default_rng(0)
N = 200_000
samples = {
    "均勻 U(0,1)": rng.random(N),
    "指數 Exp(1)": rng.exponential(1.0, N),
    "標準常態":     rng.standard_normal(N),
}
print(f"\n用 {N:,} 個樣本驗證:")
print(f"{'分佈':14s} {'樣本均值':>12} {'理論':>10} {'樣本變異數':>12} {'理論':>10}")
for name, s in samples.items():
    tm, tv = theory[name]
    print(f"{name:14s} {s.mean():12.4f} {tm:10.4f} {s.var():12.4f} {tv:10.4f}")

In [ ]:
# TODO 學生練習:加一個 Laplace 分佈 p(x) = exp(-|x|)/2
# 用 sympy 算 E[X] 與 Var(提示:偶函數/奇函數可以省一半功夫)
# 再用 rng.laplace(0, 1, N) 取樣驗證

## Lab 2｜大數法則失效:Cauchy 的樣本平均不收斂

觀念 7 說 Cauchy 沒有期望值,所以大數法則不適用。這格把常態與 Cauchy 的「累積樣本平均」畫在一起,差別非常戲劇性。


In [ ]:
rng = np.random.default_rng(1)
N = 20_000

normal = rng.standard_normal(N)
cauchy = rng.standard_cauchy(N)

# 累積平均:前 n 個樣本的平均,n = 1..N
run_normal = np.cumsum(normal) / np.arange(1, N+1)
run_cauchy = np.cumsum(cauchy) / np.arange(1, N+1)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(run_normal, lw=0.8); ax[0].axhline(0, color='C3', ls='--')
ax[0].set_title('Normal: running mean converges to 0'); ax[0].set_xscale('log')
ax[1].plot(run_cauchy, lw=0.8, color='C1'); ax[1].axhline(0, color='C3', ls='--')
ax[1].set_title('Cauchy: running mean never settles'); ax[1].set_xscale('log')
for a in ax: a.set_xlabel('n')
plt.tight_layout(); plt.show()

print(f"{'n':>8} {'常態的累積平均':>16} {'Cauchy 的累積平均':>20}")
for n in [10, 100, 1000, 10000, 20000]:
    print(f"{n:8d} {run_normal[n-1]:16.4f} {run_cauchy[n-1]:20.4f}")

print("\n常態:n 越大越貼近 0(標準差 ~ 1/sqrt(n))")
print("Cauchy:n 再大也不收斂 —— 因為 E[X] 根本不存在,大數法則不適用")
print(f"\n驗證尾巴:|x| > 100 的樣本數  常態 {np.sum(np.abs(normal)>100)}"
      f"  Cauchy {np.sum(np.abs(cauchy)>100)}")

In [ ]:
# TODO 學生練習:對 Cauchy 改用「中位數」而不是平均
# run_median = [np.median(cauchy[:n]) for n in range(1, N+1, 100)]
# 中位數會收斂嗎?為什麼中位數比平均穩健?

## Lab 3｜蒙地卡羅:誤差真的是 1/sqrt(N) 嗎

觀念 9 說蒙地卡羅誤差是 $O(N^{-1/2})$,而且與維度無關。這格在 1 維和 5 維各驗一次,看那條斜率。


In [ ]:
rng = np.random.default_rng(2)

# --- 1 維:∫_0^1 x^2 dx = 1/3 ---
exact_1d = 1/3
Ns = np.array([10**k for k in range(2, 7)])
errs_1d = []
for N in Ns:
    u = rng.random(N)
    errs_1d.append(abs(u.__pow__(2).mean() - exact_1d))

# --- 5 維:∫_[0,1]^5 (x1^2+...+x5^2) dx = 5/3 ---
exact_5d = 5/3
errs_5d = []
for N in Ns:
    u = rng.random((N, 5))
    errs_5d.append(abs((u**2).sum(axis=1).mean() - exact_5d))

print(f"{'N':>9} {'1 維誤差':>12} {'5 維誤差':>12}")
for N, e1, e5 in zip(Ns, errs_1d, errs_5d):
    print(f"{N:9d} {e1:12.3e} {e5:12.3e}")

plt.loglog(Ns, errs_1d, 'o-', label='1-D')
plt.loglog(Ns, errs_5d, 's-', label='5-D')
plt.loglog(Ns, 0.3/np.sqrt(Ns), 'k--', label=r'reference $N^{-1/2}$')
plt.xlabel('N'); plt.ylabel('|error|'); plt.legend()
plt.title('Monte Carlo error: same slope in 1-D and 5-D')
plt.show()

for name, e in [('1 維', errs_1d), ('5 維', errs_5d)]:
    slope = np.polyfit(np.log10(Ns), np.log10(e), 1)[0]
    print(f"{name} log-log 斜率 = {slope:.3f}   (理論 -0.5)")

In [ ]:
# TODO 學生練習:把維度提高到 20,誤差斜率還是 -0.5 嗎?
# 再估算:若改用 Simpson,20 維要多少格點才能達到同樣精度?(提示:n^20)